In [3]:
import pandas as pd
from pathlib import Path
import pdfplumber
import signal
import os

print('Imports OK')


Imports OK


In [5]:
base_dir = Path(".")

for label, folder in [(1, "EIA"), (0, "OUTRO")]:
    folder_path = base_dir / "DADOS_TREINAMENTO" / folder
    arquivos = sorted(folder_path.glob("*"))
    print(f"\n=== {folder} — {len(arquivos)} arquivos ===")
    for path in arquivos:
        if not path.is_file():
            continue
        tamanho_mb = path.stat().st_size / 1024 / 1024
        print(f"  Abrindo: {path.name} ({tamanho_mb:.1f} MB)", end="", flush=True)
        try:
            with pdfplumber.open(path) as pdf:
                print(f" → {len(pdf.pages)} páginas")
        except Exception as e:
            print(f" → ERRO: {e}")



=== EIA — 89 arquivos ===
  Abrindo: 01-0085-06-A-001 EIA FINAL V2.PDF (28.3 MB) → 722 páginas
  Abrindo: 0643840 - Estudo Ambiental TOMO I e II _0643840.pdf (52.4 MB) → 618 páginas
  Abrindo: 07__reas de Influ_ncia do Projeto.pdf (0.4 MB) → 4 páginas
  Abrindo: 08_Diagn_stico Ambiental_MF_Parte 01 de 03.pdf (4.7 MB) → 170 páginas
  Abrindo: 09 - Medidas e Programas.pdf (1.4 MB) → 167 páginas
  Abrindo: 1 EIA volume  I.pdf (87.7 MB) → 276 páginas
  Abrindo: 1 ESTUDO AMBIENTAL    AGOSTO 2007.pdf (0.1 MB) → 1 páginas
  Abrindo: 10 Bibliografia rev00.pdf (0.2 MB) → 15 páginas
  Abrindo: 12. Bibliografia.pdf (0.2 MB) → 60 páginas
  Abrindo: 1FRBL006-1-EA-RTE-0001-7_EIA mineroduto Ferrous - parte 1 - Caracteriza__o.pdf (6.0 MB) → 269 páginas
  Abrindo: 1FRBL006-1-SO-EIA-0001-5_EIA - parte IV - diagnostico do meio socioeconomico.pdf (9.2 MB) → 196 páginas
  Abrindo: 2 EIA  Volume II.pdf (40.5 MB) → 374 páginas
  Abrindo: 20_605_EIA_Fosnor_VOL_IIB.pdf (118.6 MB) → 784 páginas
  Abrindo: 2645

In [6]:
base_dir = Path(".")
data = []
TIMEOUT_SEGUNDOS = 120

def handler_timeout(signum, frame):
    raise TimeoutError("PDF demorou demais")

signal.signal(signal.SIGALRM, handler_timeout)

for label, folder in [(1, "EIA"), (0, "OUTRO")]:
    folder_path = base_dir / "DADOS_TREINAMENTO" / folder
    arquivos = sorted(folder_path.glob("*"))
    print(f"\n=== Processando {folder} — {len(arquivos)} arquivos ===")

    for path in arquivos:
        if not path.is_file():
            continue

        text = ""
        suffix = path.suffix.lower()
        tamanho_mb = path.stat().st_size / 1024 / 1024

        if suffix == ".pdf":
            try:
                signal.alarm(TIMEOUT_SEGUNDOS)
                with pdfplumber.open(path) as pdf:
                    pages = [page.extract_text() or "" for page in pdf.pages]
                    text = "\n".join(pages)
                signal.alarm(0)
            except TimeoutError:
                signal.alarm(0)
                print(f"  [TIMEOUT] {path.name} ({tamanho_mb:.1f} MB) — pulando")
                continue
            except Exception as e:
                signal.alarm(0)
                print(f"  [ERRO]    {path.name} — {e}")
                continue

        elif suffix == ".txt":
            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
            except Exception as e:
                print(f"  [ERRO]    {path.name} — {e}")
                continue
        else:
            print(f"  [IGNORADO] {path.name}")
            continue

        if len(text.strip()) < 100:
            print(f"  [SEM TEXTO] {path.name} ({tamanho_mb:.1f} MB) — possível PDF de imagem, pulando")
            continue

        print(f"  [OK] {path.name} ({tamanho_mb:.1f} MB) — {len(text):,} chars")
        data.append({
            "arquivo": path.name,
            "texto": text,
            "classe": label
        })

df = pd.DataFrame(data)
print(f"\n{'='*50}")
print(f"Total processado: {len(df)} documentos")
print(f"  EIA   (classe 1): {df['classe'].sum()}")
print(f"  Outro (classe 0): {(df['classe'] == 0).sum()}")
df.head()


=== Processando EIA — 89 arquivos ===
  [OK] 01-0085-06-A-001 EIA FINAL V2.PDF (28.3 MB) — 1,637,098 chars
  [OK] 0643840 - Estudo Ambiental TOMO I e II _0643840.pdf (52.4 MB) — 1,004,687 chars
  [OK] 07__reas de Influ_ncia do Projeto.pdf (0.4 MB) — 5,391 chars
  [OK] 08_Diagn_stico Ambiental_MF_Parte 01 de 03.pdf (4.7 MB) — 343,528 chars
  [OK] 09 - Medidas e Programas.pdf (1.4 MB) — 234,990 chars
  [OK] 1 EIA volume  I.pdf (87.7 MB) — 403,486 chars
  [SEM TEXTO] 1 ESTUDO AMBIENTAL    AGOSTO 2007.pdf (0.1 MB) — possível PDF de imagem, pulando
  [OK] 10 Bibliografia rev00.pdf (0.2 MB) — 34,124 chars
  [OK] 12. Bibliografia.pdf (0.2 MB) — 132,505 chars
  [OK] 1FRBL006-1-EA-RTE-0001-7_EIA mineroduto Ferrous - parte 1 - Caracteriza__o.pdf (6.0 MB) — 583,244 chars
  [OK] 1FRBL006-1-SO-EIA-0001-5_EIA - parte IV - diagnostico do meio socioeconomico.pdf (9.2 MB) — 398,132 chars
  [OK] 2 EIA  Volume II.pdf (40.5 MB) — 420,196 chars
  [OK] 20_605_EIA_Fosnor_VOL_IIB.pdf (118.6 MB) — 1,176,615 c

Could not set word spacing because /b'\x1c\xeb\xca&\xc1\xe0\x98ssIN\xc9\xae' is an invalid float value
Could not set character spacing because b'\xa6=\xa3\xe5\xa4\x90\x18\x0c\xd5\xdd\xa3\x05\x83\x811jf`\xdcZ\x9d\xc3\x90ay\xfe\x90h\x1a\xb6\xc2\xf0\x107\x00\x8c\xc36=A\x17&\x14FaVo\xcfU\xc34D\x8d\x1e~\xde`\x80(\xba\xbbkz\xc2\xaa\x81~\x80nC\x80\xa6\xb3\x83V\xd0\x9c\x86\x86NZ\xc1\x05\xb4_\x89\x1a\xcb\xae=Ak\xb4\x95\x9a\xcc\x1d\xad-c4d\xd1\xd9\x1as\xb6\xb9"\xa1\xa9id\xb0\xaf\x1b\xda[\x9a\x9b\x9a\n\xff\xbc|yd\xb0\xbf\x87\x06hkk\x8dlin\x1e\x19\x1a\xe8\xeb\xe9\xea\xec\xf8\xab\xbd\xfdJ[+\xf7\x12\xf4\xf7\xf5\xee\xfa[6\x0f\x9cI{\x9a\x1b\xe8\xef\xeb\xfd_t|\xdfP!D+\x86\x06\x07d\xd1\xdd\xf5vC\x9d\x91\x1e\x1d\x1d\x1d\x1e\x1a\xe8\xef\x8d40;i\xb4*\xac\xd7d\xbe\xe3\xa11\xe35\xeb\xcf\xdf\xd5\x9a\x03\xb6>824\x08V\xc8\xe2v}Q[p\xdd:6\x92p\xdd\x9c\xf0\xf0\xe1h\xf3\x93S\x13\xe3\xd6\xc8\x82\xda\xc9\xa9\xc8\xa7v\xcd\xce\xceLM\x8e\xdf\xad\x98\x9a\x9e\x99\x8d\xb4I\xd2l\xc0\xb4\x1c\xcf\x80\xcd&\xd9\x1d\x0e\xa9 zfz\

  [ERRO]    EA_FINAL_VOLUME_II.pdf — Invalid octal b'777' (511)
  [OK] EA_completo.pdf (69.8 MB) — 430,912 chars
  [OK] EIA   CLA   MAIO 1999.pdf (18.1 MB) — 180,388 chars
  [OK] EIA - Parte III -  Cap 6 Diag Socio ate Analise Integrada_rev.pdf (42.8 MB) — 1,110,003 chars
  [ERRO]    EIA Consolidado TBG 1997.pdf — PDF demorou demais
  [OK] EIA II.5.1.4 - Geologia e Geomormofologia.pdf (10.0 MB) — 87,935 chars
  [OK] EIA Teles Pires - Volume 1 1908_FINAL.pdf (20.5 MB) — 519,152 chars
  [OK] EIA Vol.01.pdf (17.7 MB) — 1,853,243 chars
  [OK] EIA_BIOMAR.pdf (7.1 MB) — 288,772 chars
  [ERRO]    EIA_Ecovitta_FINAL_3fev2014.pdf — PDF demorou demais
  [ERRO]    EIA_GUARDA_MOR_v6_PRINCIPAL_COM_MAPAS_E_ANEXOS_FINAL-1.pdf — PDF demorou demais
  [OK] EIA_IDENTIFICAÇÃO_EMPREENDIMENTO (4).pdf (0.2 MB) — 6,013 chars
  [OK] EIA_LT Luziania_Bsb Leste_Volume 1.pdf (13.9 MB) — 1,343,288 chars
  [OK] EIA_LT230kV_Governador_Valadares_06_Verona.pdf (66.0 MB) — 1,976,963 chars
  [OK] EIA_PORTO_MERIDIONAL_TOM

Cannot render horizontal string because [] is not a valid int, float or bytes.
Cannot render horizontal string because /'9\x0f+6\x1d:E!' is not a valid int, float or bytes.
Could not set word spacing because b'' is an invalid float value
Cannot render horizontal string because [] is not a valid int, float or bytes.
Could not set word spacing because [/b'\x9d', /b'\xae', /b'p\x93\xa1\x0f2', b'', /'5Jrx@im3', [/b'\\', /b'i\x91\x8e\x99\xc2\xbb\x7f\xaa\xa1V\x84za\x92\x88', /b'\xb0', /b'\xa7', /b'`', /b'\x90', /b'\x8b', /b'*', /b'WU,X', [9, /b"dn'N"], /b'\x06', /b"'", 6, /b'\x0f', b'76LZ/BP\n\x17$\x0b\x15#\x0e\x18(\x0e\x18&Xfr\xb4\xc8\xd1!BHDlkb\x91\x88W\x89|e\x96\x89p\xa0\x91W\x88yS\x86wx\xa9\x9aW\x86z4`X@lfj\x97\x91|\xac\xa5_\x94\x88?xiY\x94\x81Z\x96\x81:wcX\x93\x81t\xad\x9dI\x7fr\x10B8\x17JA?tlF|tM\x80yn\xa0\x9ai\x9e\x9a?us\x10<A\x05.6 HR&LZ\x04 0\x02\r\x1e$5AJgl\x86\xaa\xa8\x92\xbd\xb6Q\x85wU\x8dz^\x99\x85U\x92}H\x85qO\x87yh\x9b\x94f\x91\x93*PX>bn\x9d\xbd\xcc\x9b\xb8\xc69P]\x01\x0f\x1b\

  [OK] Microsoft Word - Relat_rio Final do Componente Ind_gena.pdf (14.0 MB) — 447,138 chars
  [OK] REL7.pdf (0.7 MB) — 192,515 chars
  [OK] RELATÓRIO EIA RIMA VOLUME_2___Cap_VI___Diagnostico_Ambiental_Meio_Fisico_Final.compressed.pdf (79.7 MB) — 612,626 chars
  [OK] SUMARIO.pdf (0.1 MB) — 28,073 chars
  [OK] TOMO III.pdf (4.0 MB) — 239,539 chars
  [OK] TOMO I_eia.pdf (21.5 MB) — 445,802 chars
  [OK] TOMO_IV_EIA_CEMDM.pdf (20.6 MB) — 444,850 chars
  [OK] TOMO_I_EIA_CEMDM.pdf (28.8 MB) — 522,630 chars
  [OK] VOLUME 1.pdf (18.0 MB) — 301,604 chars
  [OK] VOLUME 3.pdf (13.9 MB) — 851,234 chars
  [ERRO]    VOLUME_1.pdf — PDF demorou demais
  [OK] VOLUME_VI.pdf (29.3 MB) — 649,684 chars
  [OK] VOLUME_V_DIAGNOSTICO_ARQUEOLOGICo_v1.pdf (39.1 MB) — 135,179 chars
  [OK] Vol II.pdf (16.5 MB) — 107,629 chars
  [OK] Vol._2_Cap_7_ao_Anexo_7__Parte_1__Rev.01.pdf (181.7 MB) — 1,494,701 chars
  [OK] Volume 2 - TOMO I.pdf (272.0 MB) — 93,844 chars
  [OK] Volume II.pdf (1.6 MB) — 314,406 chars
  [OK] Vo

,arquivo,texto,classe
0,01-0085-06-A-001 EIA FINAL V2.PDF,MMX - MINAS - RIO MINERAÇÃO E LOGÍSTICA LTDA. ...,1
1,0643840 - Estudo Ambiental TOMO I e II _064384...,DNIT\nC G M A B\nSUMÁRIO\nTOMO I\n1. INTRODUÇÃ...,1
2,07__reas de Influ_ncia do Projeto.pdf,7. ÁREAS DE INFLUÊNCIA DO PROJETO\nA área de i...,1
3,08_Diagn_stico Ambiental_MF_Parte 01 de 03.pdf,8. DIAGNÓSTICO AMBIENTAL DA ÁREA DE INFLUÊNCIA...,1
4,09 - Medidas e Programas.pdf,Estudo de Impacto Ambiental Medidas e Programa...,1


In [7]:
df.to_csv("dataset_eia_vs_outro.csv", index=False)
print(f"Salvo: dataset_eia_vs_outro.csv ({len(df)} linhas)")

Salvo: dataset_eia_vs_outro.csv (155 linhas)


In [8]:
caminho_csv = "dataset_eia_vs_outro.csv"

df = pd.read_csv(caminho_csv)

print(df.head())
print("\nFormato:", df.shape)
print("\nDistribuição:")
print(df['classe'].value_counts())

                                             arquivo  \
0                  01-0085-06-A-001 EIA FINAL V2.PDF   
1  0643840 - Estudo Ambiental TOMO I e II _064384...   
2              07__reas de Influ_ncia do Projeto.pdf   
3     08_Diagn_stico Ambiental_MF_Parte 01 de 03.pdf   
4                       09 - Medidas e Programas.pdf   

                                               texto  classe  
0  MMX - MINAS - RIO MINERAÇÃO E LOGÍSTICA LTDA. ...       1  
1  DNIT\nC G M A B\nSUMÁRIO\nTOMO I\n1. INTRODUÇÃ...       1  
2  7. ÁREAS DE INFLUÊNCIA DO PROJETO\nA área de i...       1  
3  8. DIAGNÓSTICO AMBIENTAL DA ÁREA DE INFLUÊNCIA...       1  
4  Estudo de Impacto Ambiental Medidas e Programa...       1  

Formato: (155, 3)

Distribuição:
classe
0    79
1    76
Name: count, dtype: int64
